In [1]:
import sys, os, platform, subprocess

print('Python version:', sys.version)
print('Platform:', platform.platform())
print('Working directory:', os.getcwd())

# Install PyTorch and required build tools
!pip install -q --force-reinstall "torch==2.10.0" --index-url https://download.pytorch.org/whl/cu128
!pip install -q -U pip setuptools wheel ninja packaging openpyxl

# Install JAX and RecBole
!pip install -q --force-reinstall "jax==0.6.2" "jaxlib==0.6.2"
subprocess.run(['pip', 'install', '-q', 'numpy==1.26.4', 'recbole==1.2.0'], check=True)

# CUDA / Mamba Kernel Dependencies
!pip install -q causal-conv1d --no-build-isolation
import torch
if torch.cuda.is_available():
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    !pip install -q mamba-ssm==2.3.1 --no-build-isolation
    import mamba_ssm
    print("mamba-ssm version:", mamba_ssm.__version__)
else:
    print("No GPU detected.")

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
Working directory: /content
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.7.0 which is incompatible.
datasets 4.8.5 requires fsspec[http]<=2026.2.0,>=2023.1.0, but you have fsspec 2026.7.0 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
cuda-python 12.9.7 requires cuda-bindings~=12.9.7, but you have cuda-bindings 12.9.4 which is incompatible.
torchvision 0.26.0+cu128 requires torch==2.11.0, but you have torch 2.10.0+cu128 which is incompatible.
ERROR: pip's dependency

In [2]:
from google.colab import drive
import shutil

# Mount Google Drive
drive.mount('/content/drive')

REPO_URL  = 'https://github.com/jeromereddy9/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'
REPO_PATH = '/content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems'

if not os.path.exists(REPO_PATH):
    os.system(f'git clone {REPO_URL}')

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

print('Working Directory:', os.getcwd())

Mounted at /content/drive
Working Directory: /content/Investigating-Contrastive-Regularisation-in-State-Space-Model-Based-Sequential-Recommender-Systems


In [3]:
DRIVE_DATASET_ROOT   = '/content/drive/MyDrive/preprocessed'
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/checkpoints'
DRIVE_RESULTS_DIR    = '/content/drive/MyDrive/results'
LOCAL_DATASET_ROOT   = os.path.join(REPO_PATH, 'src/datasets/preprocessed')

os.makedirs(LOCAL_DATASET_ROOT,   exist_ok=True)
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DRIVE_RESULTS_DIR,    exist_ok=True)

DATASETS = [
    'amazon_videogames',
    'amazon_toys_and_games',
    'movielens_1m',
    'lastfm_1k'
]

for dataset in DATASETS:
    src  = os.path.join(DRIVE_DATASET_ROOT, dataset)
    dest = os.path.join(LOCAL_DATASET_ROOT, dataset)
    if os.path.exists(src) and not os.path.exists(dest):
        shutil.copytree(src, dest)
        print(f'Copied dataset locally: {dataset}')
    elif os.path.exists(dest):
        print(f'Dataset already local: {dataset}')
    else:
        print(f'Warning: Dataset source not found in Drive: {src}')

EXCEL_OUTPUT_PATH = os.path.join(DRIVE_RESULTS_DIR, 'item_coverage.xlsx')
print(f'\nCheckpoints directory: {DRIVE_CHECKPOINT_DIR}')
print(f'Excel Output path:     {EXCEL_OUTPUT_PATH}')

Copied dataset locally: amazon_videogames
Copied dataset locally: amazon_toys_and_games
Copied dataset locally: movielens_1m
Copied dataset locally: lastfm_1k

Checkpoints directory: /content/drive/MyDrive/checkpoints
Excel Output path:     /content/drive/MyDrive/results/item_coverage.xlsx


In [4]:
import warnings, logging, pickle, glob, re
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('recbole').setLevel(logging.ERROR)

from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed

from src.utils import path_builder
from src.models.Baselines.GRU4Rec import GRU4Rec
from src.models.Baselines.SASRec import SASRec
from src.models.Baselines.CL4SRec import CL4SRec
from src.models.Baselines.DouRec import DuoRec
from src.models.Baselines.mamba4rec import Mamba4Rec
from src.models.Baselines.gated_mamba import SIGMA
from src.models.SSM_CL.mamba4rec_cl import Mamba4Rec_CL
from src.models.SSM_CL.SIGMA_cl import SIGMA_CL

# Map model name strings to (model_class, config_filename)
MODEL_REGISTRY = {
    'Mamba4Rec_CL': (Mamba4Rec_CL, 'mamba4rec_cl'),
    'Mamba4Rec':    (Mamba4Rec,    'mamba4rec'),
    'SIGMA_CL':     (SIGMA_CL,     'sigma_cl'),
    'SIGMA':        (SIGMA,        'sigma'),
    'CL4SRec':      (CL4SRec,      'cl4srec'),
    'DuoRec':       (DuoRec,       'duorec'),
    'SASRec':       (SASRec,       'sasrec'),
    'GRU4Rec':      (GRU4Rec,      'gru4rec'),
}

CONFIG_DIR        = path_builder('src/configs')
DATASET_CONFIG    = os.path.join(CONFIG_DIR, 'dataset.yaml')
TRAINING_CONFIG   = os.path.join(CONFIG_DIR, 'training.yaml')
MODELS_CONFIG_DIR = os.path.join(CONFIG_DIR, 'models')

print("Model Registry registered classes:", list(MODEL_REGISTRY.keys()))

Model Registry registered classes: ['Mamba4Rec_CL', 'Mamba4Rec', 'SIGMA_CL', 'SIGMA', 'CL4SRec', 'DuoRec', 'SASRec', 'GRU4Rec']


In [5]:
def parse_checkpoint_filename(filename):
    """
    Parses checkpoint filename (without .pkl) into:
    (model_name, loss_type, cl_loss_type, dataset_name)
    Using longest-match-first strategy.
    """
    exp_id = os.path.splitext(os.path.basename(filename))[0]

    # 1. Match dataset from the end
    dataset_match = None
    for ds in sorted(DATASETS, key=len, reverse=True):
        if exp_id.endswith(ds):
            dataset_match = ds
            break

    if not dataset_match:
        return None

    prefix = exp_id[:-len(dataset_match)].rstrip('_')

    # 2. Match model name from prefix using longest-match-first
    model_match = None
    sorted_models = sorted(MODEL_REGISTRY.keys(), key=len, reverse=True)
    for m in sorted_models:
        if prefix.startswith(m):
            model_match = m
            break

    if not model_match:
        return None

    remainder = prefix[len(model_match):].strip('_')

    # Parse loss_type and cl_loss_type from remainder
    loss_type = 'default'
    cl_loss_type = 'none'

    tokens = remainder.split('_') if remainder else []

    if 'CE' in tokens:
        loss_type = 'CE'
    elif 'BPR' in tokens:
        loss_type = 'BPR'

    if 'info_nce' in remainder:
        cl_loss_type = 'info_nce'
    elif 'dcl' in remainder:
        cl_loss_type = 'dcl'

    return {
        'exp_id': exp_id,
        'model': model_match,
        'loss_type': loss_type,
        'cl_loss_type': cl_loss_type,
        'dataset': dataset_match
    }

In [9]:
@torch.no_grad()
def evaluate_checkpoint_coverage(checkpoint_path):
    """
    Loads model checkpoint, runs full_sort_predict on test set,
    masks padding item 0, and calculates item coverage at K=5, 10, 20.
    """
    parsed = parse_checkpoint_filename(checkpoint_path)
    if not parsed:
        print(f"[SKIP] Unable to parse filename: {os.path.basename(checkpoint_path)}")
        return None

    # Load checkpoint pickle
    try:
        with open(checkpoint_path, 'rb') as f:
            ckpt = pickle.load(f)
    except Exception as e:
        print(f"[SKIP] Corrupted checkpoint file {checkpoint_path}: {e}")
        return None

    if 'config' not in ckpt:
        print(f"[SKIP] Checkpoint missing 'config' key: {parsed['exp_id']}")
        return None
    if 'model_state_dict' not in ckpt:
        print(f"[SKIP] Checkpoint missing 'model_state_dict' key: {parsed['exp_id']}")
        return None

    model_name = parsed['model']
    if model_name not in MODEL_REGISTRY:
        print(f"[SKIP] Model {model_name} not found in model registry.")
        return None

    model_class, config_file = MODEL_REGISTRY[model_name]

    # Build RecBole Configuration
    config_dict = {
        'data_path': LOCAL_DATASET_ROOT,
        'eval_batch_size': 512,  # Fixed eval batch size for uniform memory usage
    }
    if parsed['loss_type'] != 'default':
        config_dict['loss_type'] = parsed['loss_type']
    if parsed['cl_loss_type'] != 'none':
        config_dict['cl_loss_type'] = parsed['cl_loss_type']
    if parsed['loss_type'] == 'BPR':
        config_dict['train_neg_sample_args'] = {
            'distribution': 'uniform', 'sample_num': 1,
            'alpha': 1.0, 'dynamic': False, 'candidate_num': 0
        }

    model_yaml_path = os.path.join(MODELS_CONFIG_DIR, f"{config_file}.yaml")

    config = Config(
        model=model_class,
        dataset=parsed['dataset'],
        config_file_list=[DATASET_CONFIG, TRAINING_CONFIG, model_yaml_path],
        config_dict=config_dict
    )

    init_seed(config['seed'], config['reproducibility'])

    # Build dataset and load test partition
    dataset = create_dataset(config)
    _, _, test_data = data_preparation(config, dataset)

    # Instantiate model and load weights
    model = model_class(config, dataset).to(config['device'])
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    # n_items includes 0 padding index; catalog size is n_items - 1
    total_catalog_items = dataset.item_num - 1
    rec_sets = {5: set(), 10: set(), 20: set()}

    for batch in test_data:
        # RecBole eval loaders return a tuple; [0] is the Interaction object, [1] is the history index
        interaction = batch[0].to(config['device'])
        history_index = batch[1]

        # Compute scores across full item space: [batch_size, n_items]
        scores = model.full_sort_predict(interaction)

        # 1. Exclude padding token (index 0)
        scores[:, 0] = -float('inf')

        # 2. Exclude history items (items the user has already interacted with)
        if history_index is not None:
            scores[history_index[0], history_index[1]] = -float('inf')

        # Accumulate top-K items across test users
        for k in [5, 10, 20]:
            _, topk_indices = torch.topk(scores, k=k, dim=-1)
            rec_sets[k].update(topk_indices.cpu().numpy().flatten())

    return {
        'exp_id': parsed['exp_id'],
        'model': parsed['model'],
        'loss_type': parsed['loss_type'],
        'cl_loss_type': parsed['cl_loss_type'],
        'dataset': parsed['dataset'],
        'n_items': total_catalog_items,
        'coverage@5':  len(rec_sets[5])  / total_catalog_items,
        'coverage@10': len(rec_sets[10]) / total_catalog_items,
        'coverage@20': len(rec_sets[20]) / total_catalog_items,
    }

In [10]:
checkpoint_files = sorted(glob.glob(os.path.join(DRIVE_CHECKPOINT_DIR, "*.pkl")))
print(f"Found {len(checkpoint_files)} checkpoint files in {DRIVE_CHECKPOINT_DIR}.\n")

# Resume existing Excel file if available
if os.path.exists(EXCEL_OUTPUT_PATH):
    results_df = pd.read_excel(EXCEL_OUTPUT_PATH)
    processed_exp_ids = set(results_df['exp_id'].values)
    results_list = results_df.to_dict('records')
    print(f"Resuming existing session: {len(processed_exp_ids)} checkpoints already evaluated.")
else:
    results_list = []
    processed_exp_ids = set()

# Process checkpoints dataset by dataset
for dataset_name in DATASETS:
    dataset_ckpts = [
        f for f in checkpoint_files
        if parse_checkpoint_filename(f) and parse_checkpoint_filename(f)['dataset'] == dataset_name
    ]

    if not dataset_ckpts:
        continue

    print(f"\n==================================================")
    print(f" Processing Dataset: {dataset_name} ({len(dataset_ckpts)} checkpoints)")
    print(f"==================================================")

    new_results_in_dataset = 0

    for ckpt_path in tqdm(dataset_ckpts, desc=f"Evaluating {dataset_name}"):
        exp_id = os.path.splitext(os.path.basename(ckpt_path))[0]

        if exp_id in processed_exp_ids:
            continue

        row = evaluate_checkpoint_coverage(ckpt_path)
        if row:
            results_list.append(row)
            processed_exp_ids.add(exp_id)
            new_results_in_dataset += 1
            print(f"  ✓ {exp_id} -> Cov@5: {row['coverage@5']:.4f} | Cov@10: {row['coverage@10']:.4f} | Cov@20: {row['coverage@20']:.4f}")

    # Save to Excel after completing each dataset
    if new_results_in_dataset > 0 or not os.path.exists(EXCEL_OUTPUT_PATH):
        df_to_save = pd.DataFrame(results_list)
        df_to_save.to_excel(EXCEL_OUTPUT_PATH, index=False)
        print(f"\n[SAVED] Updated results written to {EXCEL_OUTPUT_PATH}")

print("\nProcessing complete!")

Found 64 checkpoint files in /content/drive/MyDrive/checkpoints.


 Processing Dataset: amazon_videogames (16 checkpoints)


Evaluating amazon_videogames:   0%|          | 0/16 [00:00<?, ?it/s]

  ✓ CL4SRec_amazon_videogames -> Cov@5: 0.6597 | Cov@10: 0.7572 | Cov@20: 0.8391


  ✓ DuoRec_amazon_videogames -> Cov@5: 0.8431 | Cov@10: 0.8994 | Cov@20: 0.9439


  ✓ GRU4Rec_amazon_videogames -> Cov@5: 0.8821 | Cov@10: 0.9479 | Cov@20: 0.9798


...

  ✓ SIGMA_CE_lastfm_1k -> Cov@5: 0.1152 | Cov@10: 0.1898 | Cov@20: 0.2921


  ✓ SIGMA_CL_BPR_dcl_lastfm_1k -> Cov@5: 0.0494 | Cov@10: 0.0826 | Cov@20: 0.1322


  ✓ SIGMA_CL_BPR_info_nce_lastfm_1k -> Cov@5: 0.0532 | Cov@10: 0.0895 | Cov@20: 0.1427


  ✓ SIGMA_CL_CE_dcl_lastfm_1k -> Cov@5: 0.0986 | Cov@10: 0.1625 | Cov@20: 0.2528


  ✓ SIGMA_CL_CE_info_nce_lastfm_1k -> Cov@5: 0.0979 | Cov@10: 0.1637 | Cov@20: 0.2557

[SAVED] Updated results written to /content/drive/MyDrive/results/item_coverage.xlsx

Processing complete!


In [11]:
if os.path.exists(EXCEL_OUTPUT_PATH):
    final_df = pd.read_excel(EXCEL_OUTPUT_PATH)
    print(f"\nTotal evaluated checkpoints: {len(final_df)}")
    display(final_df.sort_values(by=['dataset', 'model', 'loss_type', 'cl_loss_type']))
else:
    print("No results file generated yet.")


Total evaluated checkpoints: 64


,exp_id,model,loss_type,cl_loss_type,dataset,n_items,coverage@5,coverage@10,coverage@20
16,CL4SRec_amazon_toys_and_games,CL4SRec,default,none,amazon_toys_and_games,75856,0.767296,0.830943,0.884228
17,DuoRec_amazon_toys_and_games,DuoRec,default,none,amazon_toys_and_games,75856,0.839077,0.888961,0.928865
18,GRU4Rec_amazon_toys_and_games,GRU4Rec,default,none,amazon_toys_and_games,75856,0.750488,0.854448,0.928773
19,Mamba4Rec_BPR_amazon_toys_and_games,Mamba4Rec,BPR,none,amazon_toys_and_games,75856,0.313027,0.409592,0.520091
20,Mamba4Rec_CE_amazon_toys_and_games,Mamba4Rec,CE,none,amazon_toys_and_games,75856,0.730450,0.828847,0.902724
...,...,...,...,...,...,...,...,...,...
43,SIGMA_CE_movielens_1m,SIGMA,CE,none,movielens_1m,3706,0.460335,0.566109,0.680518
44,SIGMA_CL_BPR_dcl_movielens_1m,SIGMA_CL,BPR,dcl,movielens_1m,3706,0.371829,0.483270,0.584997
45,SIGMA_CL_BPR_info_nce_movielens_1m,SIGMA_CL,BPR,info_nce,movielens_1m,3706,0.385051,0.486778,0.598219
46,SIGMA_CL_CE_dcl_movielens_1m,SIGMA_CL,CE,dcl,movielens_1m,3706,0.452509,0.555586,0.668915
